# GNN Final Project Notebook

This notebook runs the complete pipeline for illicit transaction detection on the Elliptic graph dataset.

If the real dataset is unavailable, it auto-uses a synthetic Elliptic-like sample so the full workflow remains executable.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('Project root:', ROOT)

Project root: /Users/arnav/Downloads/gnn_eea


In [2]:
from scripts.make_sample_data import main as make_sample_data

real_data = Path('data/elliptic')
sample_data = Path('data/sample')

if not (real_data / 'elliptic_txs_features.csv').exists():
    print('Real dataset not found at data/elliptic. Generating sample data...')
    make_sample_data()
    data_dir = sample_data
else:
    print('Using real Elliptic dataset in data/elliptic')
    data_dir = real_data

print('Data directory:', data_dir.resolve())

Real dataset not found at data/elliptic. Generating sample data...
Synthetic sample dataset written to: /Users/arnav/Downloads/gnn_eea/data/sample
Data directory: /Users/arnav/Downloads/gnn_eea/data/sample


In [3]:
from gnn_eea.config import load_config
from gnn_eea.pipeline import run_pipeline

cfg = load_config('configs/default.yaml')
cfg.data.data_dir = str(data_dir)
cfg.output_dir = 'artifacts'
cfg.training.epochs = 15
cfg.training.early_stopping_patience = 5
cfg.training.device = 'cpu'

summary = run_pipeline(cfg)
summary

    feature_mode      model  ...  selected_threshold  fpr_reduction_vs_baseline
0   all_features    xgboost  ...               0.500                        0.0
1   all_features        gcn  ...               0.485                        NaN
2   all_features  graphsage  ...               0.610                        NaN
3   all_features        gat  ...               0.625                        NaN
4  local_only_94    xgboost  ...               0.050                        0.0
5  local_only_94        gcn  ...               0.705                      -12.0
6  local_only_94  graphsage  ...               0.640                       -1.2
7  local_only_94        gat  ...               0.790                        0.6

[8 rows x 12 columns]


In [4]:
import json

metrics_dir = Path('artifacts/metrics')
for tag in ['all_features', 'local_only_94']:
    path = metrics_dir / f'dataset_metadata_{tag}.json'
    if path.exists():
        print(f'\nMetadata: {tag}')
        print(json.dumps(json.loads(path.read_text()), indent=2))


Metadata: all_features
{
  "num_nodes": 4000,
  "num_labeled": 920,
  "num_licit": 840,
  "num_illicit": 80,
  "num_unknown": 3080,
  "illicit_ratio_labeled": 0.08695652173913043,
  "all_feature_dim": 166,
  "used_feature_dim": 166,
  "num_edges": 23964,
  "train_labeled": 624,
  "val_labeled": 103,
  "test_labeled": 193
}

Metadata: local_only_94
{
  "num_nodes": 4000,
  "num_labeled": 920,
  "num_licit": 840,
  "num_illicit": 80,
  "num_unknown": 3080,
  "illicit_ratio_labeled": 0.08695652173913043,
  "all_feature_dim": 166,
  "used_feature_dim": 94,
  "num_edges": 23964,
  "train_labeled": 624,
  "val_labeled": 103,
  "test_labeled": 193
}


## Deliverables generated by notebook run

- `artifacts/metrics/results_summary.csv`
- `artifacts/metrics/results_ranked.csv`
- `artifacts/explain/*.json`
- `artifacts/models/*.pt`

Use these artifacts directly in the final report.